Let's load all the packages we will need.

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline, Pipeline
from xgboost import XGBRegressor
from category_encoders import TargetEncoder

import warnings
warnings.simplefilter("ignore")

Let's load the datasets.

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s5e4/train.csv',index_col = 'id')
test = pd.read_csv('/kaggle/input/playground-series-s5e4/test.csv',index_col='id')

In [ ]:
train = train.drop_duplicates()

Let's examine the data using head(),info() and describe().

In [ ]:
train.head()

In [ ]:
train.info()

The data has a mixture of numerical and categorical features. The train dataset has missing values in several columns. We will need to figure out how to deal with the missing values. For now, we will use a GBDT model which can handle missing values on its own so we will not need an imputation strategy. This can be adjusted later on in order to improve the model.

In [ ]:
train.describe()

Some of the percentages exceed 100 when these values should range between 0 and 100 inclusive; this could be the result of synthetic data being used. We could drop these rows as outliers but for now we will clip these values between 0 and 100. For the 'Ads' column, most of the data has between 0-2 ads but the maximum value is 103.9 which is most likely an outlier. We will need to examine the data more closely in order to deal with this. 

In [ ]:
num_cols = ['Episode_Length_minutes','Host_Popularity_percentage','Guest_Popularity_percentage','Number_of_Ads','Listening_Time_minutes']
cat_cols = ['Podcast_Name','Episode_Title','Genre','Publication_Day','Publication_Time','Episode_Sentiment']

In [ ]:
corr = train[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(data=corr,annot=True)

It is not surprising that 'Listening_Time_minutes' and 'Episode_Length_minutes' are highly postively correlated. 

In [ ]:
for col in num_cols:
    plt.figure()
    sns.histplot(data=train,x=col,bins=100)
    plt.title(f"Histogram of {col}")
    plt.show()
    plt.clf()

In [ ]:
train['Number_of_Ads'].value_counts()

We will clip the values for 'Number_of_Ads' to be between 0 and 3.

In [ ]:
for col in ['Podcast_Name','Episode_Title']:
    print(train[col].value_counts())

In [ ]:
for col in cat_cols:
    if col in ['Podcast_Name','Episode_Title']:
        continue
    plt.figure()
    sns.countplot(data=train,x=col)
    plt.title(f"Count plot of {col}")
    plt.xticks(rotation=45, ha='right')
    plt.show()
    plt.clf()

Let's see how some of these categorical features affect the target variable.

In [ ]:
for col in cat_cols:
    if col in ['Podcast_Name','Episode_Title']:
        continue
    avg_listening_time = train.groupby(col)['Listening_Time_minutes'].mean().reset_index()
    plt.figure()
    sns.lineplot(data=avg_listening_time, x=col, y='Listening_Time_minutes')    
    plt.title(f"Line plot of {col}")
    plt.xticks(rotation=45, ha='right')
    plt.show()
    plt.clf()


In [ ]:
y_train = train['Listening_Time_minutes']
X_train = train.drop('Listening_Time_minutes',axis=1)

In [ ]:
num_cols = ['Episode_Length_minutes','Host_Popularity_percentage','Guest_Popularity_percentage','Number_of_Ads']
cat_cols = ['Genre','Publication_Day', 'Publication_Time','Episode_Sentiment']
text_cols = ['Podcast_Name', 'Episode_Title'] 

def clip_ads(df):
    df['Number_of_Ads'] = np.clip(df['Number_of_Ads'],0,3)
    return df

ads_transformer = FunctionTransformer(clip_ads)

def combine_day_time(df):
    df['Publication_Day_Time'] = df['Publication_Day'] + '_' + df['Publication_Time']
    return df

day_time_transformer = make_pipeline(
    FunctionTransformer(combine_day_time,feature_names_out="one-to-one"),
    OneHotEncoder(handle_unknown='ignore')
)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

target_encoder = TargetEncoder()

preprocessor = ColumnTransformer(
    transformers=[
        ('day_time', day_time_transformer, ['Publication_Day', 'Publication_Time']),
        ('num', numerical_transformer, num_cols),
        ('podcast_name', target_encoder, 'Podcast_Name'), 
        ('episode_title', target_encoder, 'Episode_Title'),
    ],
    remainder=categorical_transformer
)


xgb_reg = XGBRegressor(random_state=42)

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', xgb_reg)])

param_grid = {
    'regressor__n_estimators': [100,200],
    'regressor__max_depth': [4,5],
    'regressor__learning_rate': [0.01, 0.1]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline,
                           param_grid,
                           cv=kf,
                           scoring='neg_mean_squared_error',
                           n_jobs=-1,
                           verbose=1,
                           return_train_score=True)

grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best Score (Negative Mean Squared Error):", grid_search.best_score_)
print("Best Score (Root Mean Squared Error):", np.sqrt(-grid_search.best_score_))

print("\nCross-Validation RMSE for each fold:")
for i in range(kf.get_n_splits()):
    fold_rmse = np.sqrt(-grid_search.cv_results_[f'split{i}_test_score'])
    print(f"Fold {i+1}: {fold_rmse.mean():.4f} (std: {fold_rmse.std():.4f})")

best_model = grid_search.best_estimator_

In [ ]:
X_test = test
test_ids = test.index

y_pred = best_model.predict(X_test)

submission = pd.DataFrame({
    'id': test_ids,
    'Listening_Time_minutes': y_pred
})


submission.to_csv("submission.csv", index=False)


In [ ]:
feature_importances = best_model.named_steps['regressor'].feature_importances_

feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("\nFeature Importance for the Best Model:")
print(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df[:20])
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()